# 🏥 Module 7: Predicting Hospital Readmissions with Gen AI + AutoML

| Duration | 45–60 minutes |
|----------|---------------|
| Objective | Use Azure OpenAI in Fabric with the OpenAI Python SDK for feature engineering, then AutoML (FLAML) to find the best readmission prediction model |
| Tools | Fabric Notebook, Fabric AI Services, PySpark, FLAML AutoML, MLflow |

---

## Why Predict Hospital Readmissions?

| Metric | Value |
|--------|-------|
| Annual cost of readmissions (US) | **$26 billion** |
| CMS penalty for excess readmissions | Up to **3% of total Medicare payments** |
| Average 30-day readmission rate | **15–20%** |
| Preventable readmissions | **27–50%** of all readmissions |

The **CMS Hospital Readmissions Reduction Program (HRRP)** penalizes hospitals with higher-than-expected readmission rates. A predictive model identifies high-risk patients **before discharge** for targeted interventions.

## Our Approach: Gen AI + AutoML

| Approach | Time | Features | AUC-ROC (typical) |
|----------|------|----------|-----------|
| Manual (domain expert) | 2–3 weeks | 15–25 | 0.70–0.75 |
| Gen AI features + single model | 2–4 days | 30–50 | 0.78–0.82 |
| **Gen AI features + AutoML** | **Hours** | **30–50** | **0.80–0.86** |

Gen AI combines the **speed** of automation with the **clinical knowledge** embedded in the LLM's training data. AutoML then removes the guesswork of algorithm selection and hyperparameter tuning — trying dozens of configurations to find the best model.

---
## Part A: Setup and AI Endpoint Configuration

### Cell 1: Install Required Libraries
Installs OpenAI SDK, FLAML AutoML, scikit-learn, and matplotlib.

> ⚠️ **Expected**: Kernel will restart after install. Wait for restart to complete before running the next cell.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: Install Required Libraries
# ══════════════════════════════════════════════════════════════
# This installs:
#   - openai: Azure OpenAI client for Fabric-auth notebooks
#   - flaml[automl]: Fast Lightweight AutoML framework
#   - scikit-learn: ML utilities (metrics, train_test_split)
#   - matplotlib: Visualization (ROC curves, feature importance)
#
# NOTE: Kernel will restart after this cell. Wait for it to
# complete before continuing.
# ══════════════════════════════════════════════════════════════

%pip install --upgrade openai flaml[automl] scikit-learn matplotlib seaborn

print("✅ Libraries installed successfully")
print("⚠️  IMPORTANT: Restart the kernel, then continue to Cell 2")

### Cell 2: Initialize Azure OpenAI Client in Fabric

Fabric uses a **Fabric-authenticated Azure OpenAI client** — no API keys, no endpoint URLs, and no separate Azure OpenAI resource configuration in the notebook.

**Expected output:** `✅ Connection successful`

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2: Initialize Azure OpenAI Client in Fabric
# ══════════════════════════════════════════════════════════════
# Azure OpenAI in Fabric uses workspace authentication via
# get_openai_httpx_sync_client(). No API keys needed.
# ══════════════════════════════════════════════════════════════

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from synapse.ml.fabric.credentials import get_openai_httpx_sync_client
import openai
import json

# ── Azure OpenAI in Fabric (Python SDK) ───────────────────────
# No API keys or endpoints needed — Fabric handles authentication
client = openai.AzureOpenAI(
    http_client=get_openai_httpx_sync_client(),
    api_version="2025-04-01-preview",
)

MODEL_NAME = "gpt-5.1"

# Quick connectivity test
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Say 'Connection successful' if you can read this."}],
    max_completion_tokens=10
)

print(f"✅ {response.choices[0].message.content}")

---
## Part B: Gen AI-Powered Feature Engineering

### Cell 3: Ask the AI Endpoint to Suggest Predictive Features

This is the core innovation — rather than manually researching clinical features from published readmission risk models (LACE index, HOSPITAL score, PARR-30), we ask the AI endpoint to analyze our data schema and suggest features grounded in medical literature.

**What happens:**
- We send our complete table schema to the LLM
- The system prompt positions it as a senior clinical data scientist
- It suggests ~25 features with computation logic, clinical rationale, and expected importance

> 💡 This takes 15-30 seconds as the LLM generates detailed feature specifications.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3: AI-Powered Feature Engineering
# ══════════════════════════════════════════════════════════════
# Instead of spending 2-3 weeks manually researching clinical
# features, we ask Azure OpenAI in Fabric to suggest features
# grounded in medical literature (LACE index, HOSPITAL score,
# Charlson Comorbidity Index, etc.)
# ══════════════════════════════════════════════════════════════

import json

# ── Describe our data schema to the LLM ────────────────────────
schema_description = """
We have a healthcare Lakehouse with these tables:

1. silver_patients: patient_id, first_name, last_name, date_of_birth, age (int),
   age_group (18-29/30-44/45-59/60-74/75+), gender (M/F), race, zip_code, city,
   state, insurance_type (Medicare/Medicaid/Commercial/Self-Pay),
   primary_care_provider, risk_score (float 0-5), risk_category (Low/Medium/High)

2. silver_encounters: encounter_id, patient_id, encounter_date (date),
   discharge_date (date), encounter_type (Inpatient/ED/Outpatient/Ambulatory),
   facility_name, department, primary_diagnosis_code (ICD-10),
   primary_diagnosis_description, attending_provider, discharge_disposition
   (Home/SNF/Home Health/Expired/Against Medical Advice),
   length_of_stay_days (int), total_charges (float), encounter_month,
   encounter_year, encounter_quarter, day_of_week (1-7), is_weekend (bool),
   los_category (Same Day/Short/Medium/Long/Extended)

3. silver_conditions: patient_id, condition_code (ICD-10), condition_description,
   condition_type (Chronic/Acute), date_diagnosed (date),
   condition_category (Diabetes/Heart Failure/COPD/Hypertension/CKD/
   Hyperlipidemia/Depression/Asthma/CAD/Obesity/Other)

4. silver_claims: encounter_id, patient_id, claim_date, claim_amount (float),
   paid_amount (float), denied_amount (float), patient_responsibility (float),
   payer, claim_status (Paid/Denied/Pending), days_to_payment (int),
   payment_ratio (float 0-1), is_denied (bool)

5. silver_medications: patient_id, medication_name, dosage, frequency,
   prescribing_provider, start_date, end_date

6. silver_vitals: patient_id, encounter_id, timestamp, heart_rate (int),
   systolic_bp (int), diastolic_bp (int), temperature_f (float),
   respiratory_rate (int), spo2_percent (int), pain_level (int 0-10),
   is_sirs_positive (bool)

7. gold_readmissions: index_encounter_id, patient_id, index_admission_date,
   index_discharge_date, index_diagnosis_code, index_diagnosis,
   index_facility, index_provider, index_los, index_disposition,
   readmit_encounter_id, readmit_date, was_readmitted (bool),
   days_to_readmission (int)
"""

system_prompt = """You are a senior clinical data scientist specializing in
hospital readmission prediction. You have deep knowledge of validated readmission
risk models (LACE index, HOSPITAL score, PARR-30, Yale/CMS model) and published
predictive features from medical literature.

Given the database schema below, suggest 25 predictive features for a 30-day
hospital readmission model. For each feature, provide:

Return a JSON array with this structure:
[
  {
    "feature_name": "descriptive_snake_case_name",
    "category": "one of: Demographics, Comorbidity, Utilization, Clinical, Financial, Temporal",
    "computation": "Brief PySpark-style pseudocode showing how to compute from our tables",
    "clinical_rationale": "Why this feature predicts readmission (cite evidence if possible)",
    "expected_importance": "high/medium/low"
  }
]

Focus on features that:
1. Are computable from the tables described (don't hallucinate columns)
2. Have clinical evidence supporting their predictive value
3. Span multiple feature categories (not all demographics)
4. Include interaction features and temporal patterns

Return valid JSON only, no other text."""

print("🤖 Asking Azure OpenAI (Fabric-authenticated client) for feature suggestions...")
print("   (This may take 15-30 seconds)\n")

try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Schema:\n{schema_description}\n\nSuggest 25 readmission features."}
        ],
        max_completion_tokens=4000,
        temperature=0.3
    )

    result_text = response.choices[0].message.content.strip()

    # Remove markdown fences
    if result_text.startswith("```json"):
        result_text = result_text[7:]
    if result_text.startswith("```"):
        result_text = result_text[3:]
    if result_text.endswith("```"):
        result_text = result_text[:-3]
    result_text = result_text.strip()

    # Replace literal backslash-n sequences with spaces
    result_text = result_text.replace('\\n', ' ')

    # Parse JSON
    ai_features = json.loads(result_text)

    print(f"✅ Successfully parsed {len(ai_features)} features")

    # Trim to exactly 25 features (prioritize high importance)
    if len(ai_features) > 25:
        importance_order = {"high": 0, "medium": 1, "low": 2}
        ai_features = sorted(
            ai_features,
            key=lambda x: importance_order.get(x.get("expected_importance", "").lower(), 3)
        )
        ai_features = ai_features[:25]
        print("   -> Trimmed to 25 features (keeping high-priority ones)\n")
    else:
        print()

except json.JSONDecodeError as e:
    print(f"❌ JSON Parse Error at Line {e.lineno}, Column {e.colno}: {e.msg}")
    start = max(0, e.pos - 100)
    print(f"   Context: ...{result_text[start:e.pos+100]}...\n")
    ai_features = []

except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}\n")
    ai_features = []

# ── Display Results ────────────────────────────────────────────
if ai_features:
    print(f"{'='*70}")
    print(f"✅ READMISSION PREDICTION FEATURES ({len(ai_features)} total)")
    print(f"{'='*70}\n")

    print(f"{'#':<3} {'Feature':<40} {'Category':<15} {'Importance':<10}")
    print("─" * 70)
    for i, feat in enumerate(ai_features, 1):
        print(f"{i:<3} {feat['feature_name']:<40} {feat['category']:<15} {feat['expected_importance']:<10}")

    print(f"\n{'='*70}")
    print("📊 FEATURES BY CATEGORY:")
    print(f"{'='*70}\n")

    categories = {}
    for feat in ai_features:
        cat = feat['category']
        categories[cat] = categories.get(cat, 0) + 1

    for cat, count in sorted(categories.items()):
        importance_high = len([f for f in ai_features if f['category'] == cat and f['expected_importance'] == 'high'])
        print(f"  {cat:<20} {count:>2} features ({importance_high} high priority)")

    print(f"\n{'='*70}")
    print("📋 DETAILED CLINICAL RATIONALE (Top 5 by importance):")
    print(f"{'='*70}\n")

    high_importance = [f for f in ai_features if f['expected_importance'] == 'high']
    for feat in high_importance[:5]:
        print(f"🔹 {feat['feature_name']}")
        print(f"   Category: {feat['category']}")
        print(f"   Computation: {feat['computation'][:100]}...")
        print(f"   Rationale: {feat['clinical_rationale']}")
        print()

    with open("readmission_features.json", "w") as f:
        json.dump(ai_features, f, indent=2)

    print(f"{'='*70}")
    print("✅ All 25 features saved to: readmission_features.json")
    print(f"{'='*70}")
else:
    print("⚠️  No features generated. Verify Azure OpenAI client and MODEL_NAME.")

### Cell 4: Build the Feature Training Dataset (42 Features)

Now we implement the AI-suggested features using PySpark, building across **6 clinical categories**:

| Category | Features | Source Table | Clinical Basis |
|----------|----------|-------------|----------------|
| **Demographics** | age, gender, insurance, risk_score | silver_patients | LACE index "A" component |
| **Comorbidity** | chronic_condition_count, has_diabetes, has_chf, etc. | silver_conditions | Charlson Comorbidity Index |
| **Prior Utilization** | prior_admissions_12m, prior_ed_visits_12m | silver_encounters | #1 predictor in published models |
| **Clinical** | SIRS count, avg heart rate, max pain | silver_vitals | Clinical instability markers |
| **Financial** | payment_ratio, denied_claims_count | silver_claims | Care complexity proxy |
| **Temporal** | is_weekend_discharge, discharge_month | gold_readmissions | "Weekend effect" in literature |
| **Medications** | medication count, is_polypharmacy | silver_medications | Drug interaction risk |

> ⚠️ Make sure your **HealthcareLakehouse** is attached in the Explorer pane before running this cell.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: Build Feature Training Dataset
# ══════════════════════════════════════════════════════════════
# This cell implements 42 features across 6 clinical categories,
# guided by the AI's suggestions and validated against published
# readmission risk models (LACE, HOSPITAL score, Charlson CCI).
#
# Output: gold_readmission_training table in the Lakehouse
# ══════════════════════════════════════════════════════════════

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ── Load base tables ────────────────────────────────────────────
readmissions = spark.table("gold_readmissions")
patients = spark.table("silver_patients")
encounters = spark.table("silver_encounters")
conditions = spark.table("silver_conditions")
claims = spark.table("silver_claims")
vitals = spark.table("silver_vitals")
medications = spark.table("silver_medications")

print(f"Base: {readmissions.count()} index admissions")
print(f"  Readmitted: {readmissions.filter(col('was_readmitted') == True).count()}")
print(f"  Not readmitted: {readmissions.filter(col('was_readmitted') == False).count()}")

# ═══════════════════════════════════════════════════════════════
# CATEGORY 1: DEMOGRAPHICS (from silver_patients)
# ═══════════════════════════════════════════════════════════════
# These features align with the "L" (Length of stay) and
# "A" (Acuity) components of the LACE readmission risk index.

# Join patient demographics
base_df = readmissions.join(
    patients.select(
        "patient_id", "age", "gender", "insurance_type",
        "risk_score", "risk_category", "race"
    ),
    "patient_id", "left"
)

# Encode categorical variables as numeric for ML
base_df = base_df \
    .withColumn("is_male", when(col("gender") == "M", 1).otherwise(0)) \
    .withColumn("is_medicare", when(col("insurance_type") == "Medicare", 1).otherwise(0)) \
    .withColumn("is_medicaid", when(col("insurance_type") == "Medicaid", 1).otherwise(0)) \
    .withColumn("is_self_pay", when(col("insurance_type") == "Self-Pay", 1).otherwise(0)) \
    .withColumn("is_high_risk", when(col("risk_category") == "High", 1).otherwise(0))

print("✓ Demographics features added")

# ═══════════════════════════════════════════════════════════════
# CATEGORY 2: COMORBIDITY (from silver_conditions)
# ═══════════════════════════════════════════════════════════════
# Comorbidity burden is the strongest predictor of readmission.
# The Charlson Comorbidity Index (CCI) uses a similar approach:
# count specific chronic conditions and score severity.

# Count chronic conditions per patient
condition_features = conditions \
    .filter(col("condition_type") == "Chronic") \
    .groupBy("patient_id") \
    .agg(
        count("*").alias("chronic_condition_count"),
        collect_set("condition_category").alias("cond_list")
    )

# Create boolean flags for key comorbidities
condition_features = condition_features \
    .withColumn("has_diabetes", array_contains(col("cond_list"), "Diabetes").cast("int")) \
    .withColumn("has_chf", array_contains(col("cond_list"), "Heart Failure").cast("int")) \
    .withColumn("has_copd", array_contains(col("cond_list"), "COPD").cast("int")) \
    .withColumn("has_hypertension", array_contains(col("cond_list"), "Hypertension").cast("int")) \
    .withColumn("has_ckd", array_contains(col("cond_list"), "Chronic Kidney Disease").cast("int")) \
    .withColumn("has_depression", array_contains(col("cond_list"), "Depression").cast("int")) \
    .withColumn("has_cad", array_contains(col("cond_list"), "Coronary Artery Disease").cast("int")) \
    .withColumn("has_obesity", array_contains(col("cond_list"), "Obesity").cast("int")) \
    .withColumn("comorbidity_cluster_count",
        # Count high-risk clusters: cardiometabolic triad, cardiopulmonary
        (array_contains(col("cond_list"), "Diabetes").cast("int") +
         array_contains(col("cond_list"), "Hypertension").cast("int") +
         array_contains(col("cond_list"), "Chronic Kidney Disease").cast("int") +
         array_contains(col("cond_list"), "Heart Failure").cast("int") +
         array_contains(col("cond_list"), "COPD").cast("int"))) \
    .drop("cond_list")

base_df = base_df.join(condition_features, "patient_id", "left")
# Fill nulls with 0 for patients without chronic conditions
for c in ["chronic_condition_count", "has_diabetes", "has_chf", "has_copd",
           "has_hypertension", "has_ckd", "has_depression", "has_cad",
           "has_obesity", "comorbidity_cluster_count"]:
    base_df = base_df.withColumn(c, coalesce(col(c), lit(0)))

print("✓ Comorbidity features added")

# ═══════════════════════════════════════════════════════════════
# CATEGORY 3: PRIOR UTILIZATION (from silver_encounters)
# ═══════════════════════════════════════════════════════════════
# Prior utilization is the strongest single predictor of future
# utilization. "The best predictor of a readmission is a prior
# readmission." (Donzé et al., JAMA Internal Medicine, 2016)

# For each index admission, count prior encounters in the last
# 12 months BEFORE the index admission date.
all_encounters = encounters.select(
    "patient_id", "encounter_id", "encounter_date", "encounter_type",
    "length_of_stay_days", "total_charges"
)

# Self-join: match each index admission to prior encounters
prior_util = readmissions.alias("idx").join(
    all_encounters.alias("prior"),
    (col("idx.patient_id") == col("prior.patient_id")) &
    (col("prior.encounter_date") < col("idx.index_admission_date")) &
    (datediff(col("idx.index_admission_date"), col("prior.encounter_date")) <= 365),
    "left"
)

# Aggregate prior utilization by index encounter
utilization_features = prior_util.groupBy("idx.index_encounter_id").agg(
    count("prior.encounter_id").alias("prior_encounters_12m"),
    sum(when(col("prior.encounter_type") == "Inpatient", 1).otherwise(0)).alias("prior_admissions_12m"),
    sum(when(col("prior.encounter_type") == "ED", 1).otherwise(0)).alias("prior_ed_visits_12m"),
    sum(when(col("prior.encounter_type") == "Outpatient", 1).otherwise(0)).alias("prior_outpatient_12m"),
    coalesce(sum("prior.total_charges"), lit(0)).alias("prior_total_charges_12m"),
    coalesce(avg("prior.length_of_stay_days"), lit(0)).alias("prior_avg_los")
)

base_df = base_df.join(utilization_features, 
    base_df["index_encounter_id"] == utilization_features["index_encounter_id"], "left") \
    .drop(utilization_features["index_encounter_id"])

# Fill nulls (patients with no prior encounters)
for c in ["prior_encounters_12m", "prior_admissions_12m", "prior_ed_visits_12m",
           "prior_outpatient_12m", "prior_total_charges_12m", "prior_avg_los"]:
    base_df = base_df.withColumn(c, coalesce(col(c), lit(0)))

print("✓ Prior utilization features added")

# ═══════════════════════════════════════════════════════════════
# CATEGORY 4: CLINICAL (from silver_vitals)
# ═══════════════════════════════════════════════════════════════
# Vital sign patterns during the index stay indicate clinical
# instability. SIRS-positive readings suggest underlying infection
# or inflammatory response — a strong readmission risk factor.

# Aggregate vitals per encounter (for the index admission)
vitals_features = vitals.groupBy("encounter_id").agg(
    count("*").alias("vitals_reading_count"),
    sum(col("is_sirs_positive").cast("int")).alias("sirs_positive_count"),
    avg("heart_rate").alias("avg_heart_rate"),
    avg("temperature_f").alias("avg_temperature"),
    avg("respiratory_rate").alias("avg_respiratory_rate"),
    avg("spo2_percent").alias("avg_spo2"),
    max("pain_level").alias("max_pain_level"),
    max("heart_rate").alias("max_heart_rate"),
    min("spo2_percent").alias("min_spo2")
)

base_df = base_df.join(vitals_features,
    base_df["index_encounter_id"] == vitals_features["encounter_id"], "left") \
    .drop(vitals_features["encounter_id"])

# Fill nulls
for c in ["vitals_reading_count", "sirs_positive_count", "avg_heart_rate",
           "avg_temperature", "avg_respiratory_rate", "avg_spo2",
           "max_pain_level", "max_heart_rate", "min_spo2"]:
    base_df = base_df.withColumn(c, coalesce(col(c), lit(0)))

print("✓ Clinical vitals features added")

# ═══════════════════════════════════════════════════════════════
# CATEGORY 5: FINANCIAL (from silver_claims)
# ═══════════════════════════════════════════════════════════════
# Financial patterns correlate with care complexity. Denied claims
# may indicate documentation gaps or coding issues that affect
# continuity of care.

claims_features = claims.groupBy("encounter_id").agg(
    sum("claim_amount").alias("claim_total_amount"),
    avg("payment_ratio").alias("avg_payment_ratio"),
    sum(col("is_denied").cast("int")).alias("denied_claims_count"),
    avg("days_to_payment").alias("avg_days_to_payment")
)

base_df = base_df.join(claims_features,
    base_df["index_encounter_id"] == claims_features["encounter_id"], "left") \
    .drop(claims_features["encounter_id"])

for c in ["claim_total_amount", "avg_payment_ratio", "denied_claims_count", "avg_days_to_payment"]:
    base_df = base_df.withColumn(c, coalesce(col(c), lit(0)))

print("✓ Financial features added")

# ═══════════════════════════════════════════════════════════════
# CATEGORY 6: TEMPORAL (from gold_readmissions)
# ═══════════════════════════════════════════════════════════════
# The "weekend effect" — patients discharged on weekends have
# higher readmission rates due to reduced post-discharge support.

base_df = base_df \
    .withColumn("discharge_day_of_week", dayofweek(col("index_discharge_date"))) \
    .withColumn("is_weekend_discharge",
        when(dayofweek(col("index_discharge_date")).isin(1, 7), 1).otherwise(0)) \
    .withColumn("discharge_month", month(col("index_discharge_date"))) \
    .withColumn("discharge_quarter", quarter(col("index_discharge_date"))) \
    .withColumn("is_snf_discharge",
        when(col("index_disposition") == "SNF", 1).otherwise(0)) \
    .withColumn("is_home_health_discharge",
        when(col("index_disposition") == "Home Health", 1).otherwise(0))

print("✓ Temporal features added")

# ═══════════════════════════════════════════════════════════════
# MEDICATION FEATURES (from silver_medications)
# ═══════════════════════════════════════════════════════════════
# Polypharmacy (5+ medications) is a strong readmission risk
# factor — it increases drug interaction risk and non-adherence.

med_features = medications.groupBy("patient_id").agg(
    countDistinct("medication_name").alias("unique_medication_count")
)
med_features = med_features.withColumn(
    "is_polypharmacy",
    when(col("unique_medication_count") >= 5, 1).otherwise(0)
)

base_df = base_df.join(med_features, "patient_id", "left")
base_df = base_df \
    .withColumn("unique_medication_count", coalesce(col("unique_medication_count"), lit(0))) \
    .withColumn("is_polypharmacy", coalesce(col("is_polypharmacy"), lit(0)))

print("✓ Medication features added")

# ═══════════════════════════════════════════════════════════════
# CREATE TARGET VARIABLE & FEATURE MATRIX
# ═══════════════════════════════════════════════════════════════
# Select only numeric feature columns + target variable

feature_columns = [
    # Demographics
    "age", "is_male", "risk_score", "is_medicare", "is_medicaid",
    "is_self_pay", "is_high_risk",
    # Comorbidity
    "chronic_condition_count", "has_diabetes", "has_chf", "has_copd",
    "has_hypertension", "has_ckd", "has_depression", "has_cad",
    "has_obesity", "comorbidity_cluster_count",
    # Prior Utilization
    "prior_encounters_12m", "prior_admissions_12m", "prior_ed_visits_12m",
    "prior_outpatient_12m", "prior_total_charges_12m", "prior_avg_los",
    # Clinical
    "index_los", "vitals_reading_count", "sirs_positive_count",
    "avg_heart_rate", "avg_temperature", "avg_respiratory_rate",
    "avg_spo2", "max_pain_level", "max_heart_rate", "min_spo2",
    # Financial
    "claim_total_amount", "avg_payment_ratio", "denied_claims_count",
    "avg_days_to_payment",
    # Temporal
    "is_weekend_discharge", "discharge_month", "discharge_quarter",
    "is_snf_discharge", "is_home_health_discharge",
    # Medications
    "unique_medication_count", "is_polypharmacy"
]

# Cast target to integer (0/1)
training_df = base_df.withColumn("label", col("was_readmitted").cast("int")) \
    .select(feature_columns + ["label", "index_encounter_id", "patient_id"])

# Cast all feature columns to double for ML compatibility
for c in feature_columns:
    training_df = training_df.withColumn(c, col(c).cast("double"))

# Fill any remaining nulls with 0
training_df = training_df.na.fill(0.0)

print(f"\n📊 Training dataset: {training_df.count()} rows × {len(feature_columns)} features")
readmit_count = training_df.filter(col("label") == 1).count()
no_readmit = training_df.filter(col("label") == 0).count()
print(f"   Label distribution: {readmit_count} readmitted ({readmit_count/(readmit_count+no_readmit)*100:.1f}%), "
      f"{no_readmit} not readmitted ({no_readmit/(readmit_count+no_readmit)*100:.1f}%)")

# Save the feature-engineered dataset to the Gold layer
training_df.write.mode("overwrite").format("delta").saveAsTable("gold_readmission_training")
print("✓ Saved to gold_readmission_training")

---
## Part C: AutoML — Automatically Find the Best Model

### Cell 5: Run AutoML with MLflow Tracking

**FLAML AutoML** removes the guesswork of algorithm selection and hyperparameter tuning:

| Algorithm | Strengths |
|-----------|----------|
| **LightGBM** | Fast, handles categorical features natively |
| **XGBoost** | Robust, handles missing values |
| **CatBoost** | Excellent with categorical features |
| **Random Forest** | Stable, resistant to overfitting |
| **Extra Trees** | Faster than Random Forest |

FLAML will try all of these within a 2-minute time budget and return the best one.

> ⏱️ This cell takes ~2 minutes to run (the configured time budget).

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: Run AutoML with MLflow Experiment Tracking
# ══════════════════════════════════════════════════════════════
# FLAML (Fast Lightweight AutoML) will:
#   1. Try multiple algorithms (LightGBM, XGBoost, CatBoost, RF, ET)
#   2. Intelligently search hyperparameter space
#   3. Use cross-validation to prevent overfitting
#   4. Return the best model within our time budget
#
# MLflow automatically logs every trial for reproducibility.
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, average_precision_score
)
from flaml import AutoML
import mlflow
import matplotlib.pyplot as plt
%matplotlib inline

# ── Load feature-engineered data ────────────────────────────
feature_columns = [
    "age", "is_male", "risk_score", "is_medicare", "is_medicaid",
    "is_self_pay", "is_high_risk",
    "chronic_condition_count", "has_diabetes", "has_chf", "has_copd",
    "has_hypertension", "has_ckd", "has_depression", "has_cad",
    "has_obesity", "comorbidity_cluster_count",
    "prior_encounters_12m", "prior_admissions_12m", "prior_ed_visits_12m",
    "prior_outpatient_12m", "prior_total_charges_12m", "prior_avg_los",
    "index_los", "vitals_reading_count", "sirs_positive_count",
    "avg_heart_rate", "avg_temperature", "avg_respiratory_rate",
    "avg_spo2", "max_pain_level", "max_heart_rate", "min_spo2",
    "claim_total_amount", "avg_payment_ratio", "denied_claims_count",
    "avg_days_to_payment",
    "is_weekend_discharge", "discharge_month", "discharge_quarter",
    "is_snf_discharge", "is_home_health_discharge",
    "unique_medication_count", "is_polypharmacy"
]

# Load from Gold table
training_spark_df = spark.table("gold_readmission_training")
pandas_df = training_spark_df.select(feature_columns + ["label"]).toPandas()

X = pandas_df[feature_columns]
y = pandas_df["label"]

print(f"Dataset: {len(X)} samples, {len(feature_columns)} features")
print(f"Label distribution: {y.sum()} readmitted ({y.mean()*100:.1f}%), "
      f"{len(y)-y.sum()} not readmitted ({(1-y.mean())*100:.1f}%)")

# ── Train/Test Split (80/20, stratified) ────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train)} samples | Test: {len(X_test)} samples")

# ── Configure and run AutoML ─────────────────────────────────
automl = AutoML()

automl_settings = {
    "time_budget": 120,          # 2 minutes search budget
    "metric": "roc_auc",         # Optimize for AUC-ROC (healthcare standard)
    "task": "classification",    # Binary classification
    "estimator_list": [          # Algorithms to search over
        "lgbm",                  # LightGBM
        "xgboost",               # XGBoost
        "catboost",              # CatBoost
        "rf",                    # Random Forest
        "extra_tree",            # Extra Trees
    ],
    "log_file_name": "automl_readmission.log",
    "seed": 42,
    "verbose": 1,
}

print(f"\n{'='*60}")
print(f"🔬 STARTING AUTOML (FLAML)")
print(f"{'='*60}")
print(f"  Time budget: {automl_settings['time_budget']} seconds")
print(f"  Metric: {automl_settings['metric']}")
print(f"  Algorithms: {', '.join(automl_settings['estimator_list'])}")
print(f"  FLAML will automatically search for the best model...")
print(f"{'='*60}\n")

# Enable MLflow autologging to track all trials
mlflow.autolog(exclusive=False)

with mlflow.start_run(run_name="AutoML_Readmission_Prediction"):
    automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# ── Display AutoML Results ────────────────────────────────────
print(f"\n{'='*60}")
print(f"🏆 AUTOML RESULTS")
print(f"{'='*60}")
print(f"  Best algorithm:      {automl.best_estimator}")
print(f"  Best AUC-ROC (CV):   {1 - automl.best_loss:.4f}")
print(f"  Training time:       {automl.best_config_train_time:.1f} seconds")
print(f"  Total trials:        {len(automl.config_history)}")
print(f"\n  Best hyperparameters:")
for param, value in automl.best_config.items():
    print(f"    {param}: {value}")

# ── Evaluate on Hold-out Test Set ─────────────────────────────
y_prob = automl.predict_proba(X_test)[:, 1]
y_pred = automl.predict(X_test)

auc_roc = roc_auc_score(y_test, y_prob)
avg_precision = average_precision_score(y_test, y_prob)

print(f"\n{'='*60}")
print(f"📊 TEST SET EVALUATION (hold-out)")
print(f"{'='*60}")
print(f"  AUC-ROC:            {auc_roc:.4f}")
print(f"  Average Precision:  {avg_precision:.4f}")
print(f"\n  Interpretation:")
if auc_roc >= 0.80:
    print(f"  ✅ Excellent discriminative ability (≥0.80)")
elif auc_roc >= 0.75:
    print(f"  ✅ Good discriminative ability (0.75-0.80)")
elif auc_roc >= 0.70:
    print(f"  ⚠️ Acceptable but could improve (0.70-0.75)")
else:
    print(f"  ⚠️ Needs improvement (<0.70) — try more features or data")

print(f"\n{'='*60}")
print(f"📋 CLASSIFICATION REPORT")
print(f"{'='*60}")
print(classification_report(y_test, y_pred, target_names=["Not Readmitted", "Readmitted"]))

# ── Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:")
print(f"  True Negatives:  {cm[0][0]:4d}  |  False Positives: {cm[0][1]:4d}")
print(f"  False Negatives: {cm[1][0]:4d}  |  True Positives:  {cm[1][1]:4d}")

# ── Feature Importance ──────────────────────────────────────
best_model = automl.model.estimator
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
    feat_importance = sorted(zip(feature_columns, importance), key=lambda x: x[1], reverse=True)

    print(f"\n{'='*60}")
    print(f"🔑 TOP 15 FEATURES BY IMPORTANCE ({automl.best_estimator})")
    print(f"{'='*60}")
    for i, (feat, imp) in enumerate(feat_importance[:15], 1):
        bar = "█" * int(imp / feat_importance[0][1] * 30)
        print(f"  {i:2d}. {feat:<35s} {imp:.4f}  {bar}")
else:
    print("\n  (Feature importance not available for this model type)")
    feat_importance = list(zip(feature_columns, [0.0] * len(feature_columns)))

### Cell 6: Visualize Model Performance

Three key visualizations for stakeholder presentations:
1. **ROC Curve** — Gold standard for classifier evaluation
2. **Precision-Recall Curve** — More informative for imbalanced datasets
3. **Feature Importance** — "WHY does the model think this patient is high-risk?"

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6: Visualize Model Performance
# ══════════════════════════════════════════════════════════════
# Three charts that tell the complete model story:
#   1. ROC Curve — overall discriminative ability
#   2. Precision-Recall — performance on the minority class
#   3. Feature Importance — what drives predictions
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── Plot 1: ROC Curve ─────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'{automl.best_estimator} (AUC = {auc_roc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('ROC Curve — Readmission Prediction (AutoML Best)')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Precision-Recall Curve ────────────────────────────
precision, recall, _ = precision_recall_curve(y_test, y_prob)
axes[1].plot(recall, precision, 'r-', linewidth=2,
             label=f'{automl.best_estimator} (AP = {avg_precision:.3f})')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision (PPV)')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# ── Plot 3: Top 15 Feature Importance ─────────────────────────
if feat_importance[0][1] > 0:
    top15 = feat_importance[:15]
    top15_names = [f[0].replace("_", " ").title() for f in reversed(top15)]
    top15_values = [f[1] for f in reversed(top15)]
    colors = ['#e74c3c' if v > 0.05 else '#3498db' for v in top15_values]
    axes[2].barh(range(len(top15_names)), top15_values, color=colors)
    axes[2].set_yticks(range(len(top15_names)))
    axes[2].set_yticklabels(top15_names, fontsize=9)
    axes[2].set_xlabel('Feature Importance (Gain)')
    axes[2].set_title(f'Top 15 Features ({automl.best_estimator})')
    axes[2].grid(True, alpha=0.3, axis='x')
else:
    axes[2].text(0.5, 0.5, "Feature importance\nnot available",
                 ha='center', va='center', fontsize=14)
    axes[2].set_title('Feature Importance')

plt.tight_layout()
display(fig)
print("📈 Plots generated — see above for ROC curve, PR curve, and feature importance")

### Cell 7: AutoML Trial Summary & Comparison to Published Models

Compare our AutoML model against validated clinical readmission risk models.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7: AutoML Trial History & Benchmark Comparison
# ══════════════════════════════════════════════════════════════
# Compare our model against published readmission risk scores
# used in clinical practice today.
# ══════════════════════════════════════════════════════════════

# ── Show all trials that AutoML evaluated ─────────────────────
print(f"{'='*70}")
print(f"🔬 AUTOML TRIAL HISTORY ({len(automl.config_history)} configurations tried)")
print(f"{'='*70}")
print(f"\n{'Trial':<7} {'Algorithm':<15} {'AUC-ROC (CV)':<15} {'Train Time':<12}")
print("─" * 50)

for trial_id, config in automl.config_history.items():
    if isinstance(config, dict):
        estimator = config.get("Current Learner", "unknown")
    elif isinstance(config, tuple):
        estimator = config[0] if config else "unknown"
    else:
        estimator = str(config)
    print(f"{trial_id:<7} {str(estimator):<15}")

# Compare AutoML result to baseline
print(f"\n{'='*70}")
print(f"📊 COMPARISON: AutoML vs Published Readmission Models")
print(f"{'='*70}")
print(f"  {'Model':<35} {'AUC-ROC':<12} {'Notes'}")
print(f"  {'─'*65}")
print(f"  {'LACE Index':<35} {'0.68-0.72':<12} {'Validated in >50 studies'}")
print(f"  {'HOSPITAL Score':<35} {'0.72':<12} {'7-point scale'}")
print(f"  {'Yale/CMS Model':<35} {'0.73-0.76':<12} {'Used for HRRP penalties'}")
print(f"  {'─'*65}")
best_est = str(automl.best_estimator) if automl.best_estimator else "Unknown"
print(f"  {'Our AutoML Model (' + best_est + ')':<35} {f'{auc_roc:.4f}':<12} {'AutoML-optimized, 42 features'}")
print(f"  {'─'*65}")

if auc_roc > 0.76:
    print(f"\n  ✅ Our model OUTPERFORMS published readmission risk models!")
    print(f"     This is expected: we use 42 features vs. LACE's 4 variables.")
elif auc_roc > 0.72:
    print(f"\n  ✅ Our model is on par with the best published models.")
else:
    print(f"\n  ⚠️ Consider increasing time_budget or adding more features.")

---
## Part D: Score Patients and Generate Clinical Insights

### Cell 8: Generate Risk Scores for All Patients

Classify every patient into risk tiers for clinical action:

| Risk Tier | Score Range | Clinical Action |
|-----------|-------------|----------------|
| **Low** | < 0.2 | Routine discharge, standard follow-up |
| **Medium** | 0.2 – 0.5 | Enhanced follow-up within 48 hours |
| **High** | ≥ 0.5 | Active intervention before discharge |

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8: Score All Patients & Classify Risk Tiers
# ══════════════════════════════════════════════════════════════
# Score every index admission with a readmission probability
# (0.0 to 1.0) and classify into Low/Medium/High risk tiers.
#
# Output: gold_readmission_risk_scores table in Lakehouse
# ══════════════════════════════════════════════════════════════

from pyspark.sql.functions import col, count, avg, sum, round

# Score ALL patients using the AutoML best model
X_all = pandas_df[feature_columns]
all_probs = automl.predict_proba(X_all)[:, 1]
all_preds = automl.predict(X_all)

# Load full training data from Spark to get encounter/patient IDs
full_df = training_spark_df.select(
    "index_encounter_id", "patient_id", "label", *feature_columns
).toPandas()

# Add predictions
full_df["readmission_risk_score"] = all_probs
full_df["predicted_readmission"] = all_preds
full_df["risk_tier"] = pd.cut(
    all_probs,
    bins=[0, 0.2, 0.5, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

# Convert back to Spark and save
scored_df = spark.createDataFrame(full_df[["index_encounter_id", "patient_id",
    "readmission_risk_score", "predicted_readmission", "risk_tier", "label"]])

scored_df = scored_df \
    .withColumn("readmission_risk_score", col("readmission_risk_score").cast("double")) \
    .withColumn("predicted_readmission", col("predicted_readmission").cast("int")) \
    .withColumn("actual_readmission", col("label").cast("int")) \
    .drop("label")

scored_df.write.mode("overwrite").format("delta").saveAsTable("gold_readmission_risk_scores")

# ── Display risk distribution ────────────────────────────────
print(f"✅ Saved {scored_df.count()} risk scores to gold_readmission_risk_scores\n")

print("Risk Tier Distribution:")
scored_df.groupBy("risk_tier").agg(
    count("*").alias("patients"),
    round(avg("readmission_risk_score"), 3).alias("avg_risk_score"),
    sum("actual_readmission").alias("actual_readmissions")
).orderBy("risk_tier").show()

# Show high-risk patients for care management review
print("🚨 HIGH RISK PATIENTS (top 10 by readmission risk score):")
scored_df.filter(col("risk_tier") == "High") \
    .orderBy(col("readmission_risk_score").desc()) \
    .show(10)

### Cell 9: AI-Generated Clinical Interpretation

Transform raw ML metrics into a **clinician-readable report** for hospital leadership (CMO, VP Quality). The AI endpoint bridges the gap between data science output and clinical decision-making.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9: AI-Generated Clinical Interpretation
# ══════════════════════════════════════════════════════════════
# Ask Azure OpenAI in Fabric to interpret model results for
# clinical stakeholders — transforming AUC-ROC and feature
# importance into actionable recommendations.
# ══════════════════════════════════════════════════════════════

# Build a summary of model results for the LLM
top_features_text = "\n".join([
    f"  {i+1}. {feat} (importance: {imp:.4f})"
    for i, (feat, imp) in enumerate(feat_importance[:15])
])

risk_dist = scored_df.groupBy("risk_tier").agg(
    count("*").alias("patients"),
    round(avg("readmission_risk_score"), 3).alias("avg_risk"),
    sum("actual_readmission").alias("actual_readmits")
).orderBy("risk_tier").toPandas()

risk_dist_text = risk_dist.to_string(index=False)

model_summary = f"""
We trained a readmission prediction model using AutoML (FLAML).

AutoML Trial Summary:
- Best algorithm: {automl.best_estimator}
- Total algorithms tried: LightGBM, XGBoost, CatBoost, Random Forest, Extra Trees
- Total configurations evaluated: {len(automl.config_history)}
- Best hyperparameters: {automl.best_config}

Model Performance:
- AUC-ROC: {auc_roc:.4f}
- Average Precision: {avg_precision:.4f}
- Dataset: {len(pandas_df)} inpatient index admissions
- Readmission rate: {y.mean()*100:.1f}%
- Features used: {len(feature_columns)} across 6 categories
  (Demographics, Comorbidity, Utilization, Clinical, Financial, Temporal)

Top 15 Features by Importance:
{top_features_text}

Risk Tier Distribution:
{risk_dist_text}
"""

interpretation_prompt = """You are a clinical informatics expert presenting
ML model results to a hospital's Chief Medical Officer and VP of Quality.

Given the readmission prediction model results below, provide:

1. **Clinical Interpretation** (3-4 paragraphs): Explain what the model found
   in plain language. What drives readmissions at this hospital? How do the
   top features align with published literature (LACE index, HOSPITAL score)?

2. **Actionable Recommendations** (5-7 bullet points): Based on the top
   predictive features, what specific interventions should the hospital
   implement? Be specific — name programs, staffing changes, or workflows.

3. **Model Limitations** (3-4 bullet points): What are the caveats? What
   should we tell clinicians about trusting these predictions?

4. **Comparison to Published Models**: How does this AUC-ROC compare to
   LACE (0.68-0.72), HOSPITAL score (0.72), and other published models?

5. **AutoML Advantage**: Briefly explain the value of using AutoML vs.
   manually tuning a single model. Why should the CMO trust that we found
   the best model?

Write for a clinical audience, not data scientists."""

print("🤖 Asking Azure OpenAI in Fabric to interpret model results...\n")

interpretation_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": interpretation_prompt},
        {"role": "user", "content": model_summary},
    ],
    max_completion_tokens=2500,
    temperature=0.5,
 )

clinical_report = interpretation_response.choices[0].message.content

# Display nicely formatted output
print("=" * 80)
print("🏥 CLINICAL INTERPRETATION REPORT")
print("=" * 80)
print(clinical_report)
print("=" * 80)

# Save report for sharing
with open('/lakehouse/default/Files/readmission_clinical_interpretation.md', 'w') as f:
    f.write("# Readmission Prediction Model - Clinical Interpretation\n\n")
    f.write(clinical_report)

print("\n💾 Report saved to: Files/readmission_clinical_interpretation.md")

---
## ✅ Core Module Complete!

**What you built:**
- Gen AI suggested 25+ clinically-grounded features
- 42 features engineered across 6 categories saved to `gold_readmission_training`
- AutoML (FLAML) found the best model automatically
- All patients scored and classified into risk tiers in `gold_readmission_risk_scores`
- AI-generated clinical interpretation for stakeholders

---

# 🔽 Optional Sections Below

The following sections cover advanced deployment, MLflow 3, and monitoring capabilities.

---
## Optional Part E: Deploy the Model for Real-Time Predictions

Batch scoring works for daily risk reports, but clinical workflows often need **instant predictions** (e.g., EHR discharge alerts). Fabric ML model endpoints provide:

- **Zero-setup deployment** — no Docker, no Kubernetes
- **Auto-scaling** — scales to 3 nodes under high traffic, scales to zero when idle
- **Built-in authentication** — secured with Fabric workspace permissions
- **REST API** — any application can call the endpoint

### Cell 10: Register the Best Model with MLflow

This creates a **Fabric ML Model item** in your workspace that supports versioning, comparison, and endpoint deployment.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10: Register Model for Deployment
# ══════════════════════════════════════════════════════════════
# Register the best AutoML model as a versioned Fabric ML Model
# item. This enables:
#   - Version tracking and comparison
#   - Real-time endpoint activation
#   - Batch scoring with the PREDICT function
#
# Reference: https://learn.microsoft.com/en-us/fabric/data-science/machine-learning-model
# ══════════════════════════════════════════════════════════════

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# ── Register the AutoML best model as a Fabric ML Model ──────
mlflow.set_experiment("AutoML_Readmission_Prediction")

with mlflow.start_run(run_name="Register_Best_Model") as run:
    # Infer model signature (input/output schema)
    signature = infer_signature(X_train, automl.predict(X_train))
    
    # Log the best model with input example for endpoint validation
    model_info = mlflow.sklearn.log_model(
        sk_model=automl.model,
        artifact_path="readmission_model",
        signature=signature,
        input_example=X_test.head(3),
    )
    
    # Log key metrics alongside the model
    mlflow.log_metrics({
        "auc_roc": auc_roc,
        "avg_precision": avg_precision,
        "n_features": len(feature_columns),
        "n_training_samples": len(X_train),
    })
    
    # Log model metadata as tags
    mlflow.set_tags({
        "algorithm": automl.best_estimator,
        "use_case": "30-day-readmission-prediction",
        "department": "quality-improvement",
        "automl_trials": str(len(automl.config_history)),
    })
    
    print(f"✅ Model logged — Run ID: {run.info.run_id}")
    print(f"   Model URI: {model_info.model_uri}")

# ── Register as a versioned ML Model item ─────────────────────
model_name = "Readmission_Risk_Model"
model_uri = f"runs:/{run.info.run_id}/readmission_model"

mv = mlflow.register_model(model_uri, model_name)

print(f"\n✅ Model registered in Fabric:")
print(f"   Name: {mv.name}")
print(f"   Version: {mv.version}")
print(f"   Status: {mv.status}")
print(f"\n💡 You can now find 'Readmission_Risk_Model' under your workspace items.")
print(f"\n📋 NEXT STEPS (in Fabric UI):")
print(f"   1. Open the model → select Version 1")
print(f"   2. Click 'Activate version endpoint' in the ribbon")
print(f"   3. Wait 2-3 min for status: Activating → Active")
print(f"   4. Copy the Endpoint URL for real-time scoring")

### Cell 11: Query the Endpoint / Batch Scoring with PREDICT

Two ways to consume the registered model:
1. **Real-time endpoint** — REST API for on-demand predictions (requires activation in UI)
2. **PREDICT function** — Batch scoring in Spark (no endpoint needed)

> ⚠️ The real-time endpoint must be activated in the Fabric UI before calling it. The PREDICT function works immediately after registration.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11: Consume the Deployed Model
# ══════════════════════════════════════════════════════════════
# Two approaches:
#   A) Real-time endpoint (REST API) — for on-demand scoring
#   B) PREDICT function (batch) — for scheduled scoring jobs
#
# Reference: https://learn.microsoft.com/en-us/fabric/data-science/model-endpoints
# ══════════════════════════════════════════════════════════════

import json

# ═══════════════════════════════════════════════════════════════
# APPROACH A: Real-Time Endpoint (REST API)
# ═══════════════════════════════════════════════════════════════
# After activating in the UI, the endpoint URL is:
# https://<region>.api.fabric.microsoft.com/v1/workspaces/<id>/models/<id>/versions/1/score

# Build a sample payload from test data
sample_patients = X_test.head(5).to_dict(orient="records")
payload = {"input_data": sample_patients}

print("═" * 60)
print("📡 REAL-TIME ENDPOINT (after activation in UI)")
print("═" * 60)
print("\n📋 Sample request payload (first patient):")
print(json.dumps(sample_patients[0], indent=2)[:500])

print("\n\n# Example: Calling the endpoint from any application")
print("# ─────────────────────────────────────────────────────")
print("""import requests

url = "https://<region>.api.fabric.microsoft.com/v1/workspaces/<workspace-id>/models/<model-id>/versions/1/score"
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}
response = requests.post(url, headers=headers, json=payload)
print(response.json())  # Returns prediction probability
""")

# ═══════════════════════════════════════════════════════════════
# APPROACH B: Batch Scoring with PREDICT (no endpoint needed)
# ═══════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("📊 BATCH SCORING WITH PREDICT (no endpoint needed)")
print("═" * 60)

from synapse.ml.predict import MLFlowTransformer

# Load the registered model for batch inference
model = MLFlowTransformer(
    inputCols=feature_columns,
    outputCol="prediction",
    modelName="Readmission_Risk_Model",
    modelVersion=1,
)

# Score a Spark DataFrame directly
test_spark_df = spark.createDataFrame(X_test.head(20))
predictions = model.transform(test_spark_df)
predictions.select(feature_columns[:3] + ["prediction"]).show(5)

print("✅ Batch predictions generated using registered model")
print("\n💡 Capacity note: Active endpoints consume 5 CU/s per node.")
print("   With auto-sleep ON (default), idle endpoints scale to zero after 5 min.")

---
## Optional Part F: MLflow 3 — LoggedModel and Generative AI Traces

MLflow 3 introduces two major capabilities:

| Capability | MLflow 2.x | MLflow 3 |
|-----------|-----------|----------|
| **Model logging** | Artifact attached to a run | First-class **LoggedModel** entity |
| **Gen AI observability** | Not available | **Traces** for prompts, responses, tool calls, latency, tokens |

Since this module uses Gen AI for feature engineering and result interpretation, MLflow 3 traces let you **audit every LLM call** — essential for AI-assisted clinical decisions.

### Cell 12: Install MLflow 3

> ⚠️ Kernel will restart after this install. Wait for completion before proceeding.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12: Upgrade to MLflow 3
# ══════════════════════════════════════════════════════════════
# Fabric ships MLflow 2.x by default. This upgrades to MLflow 3.1
# which adds LoggedModel entities and Gen AI trace capture.
#
# Reference: https://learn.microsoft.com/en-us/fabric/data-science/mlflow-3-overview
# ══════════════════════════════════════════════════════════════

%pip install "synapseml-mlflow[online-notebook]>=2.0.3" "mlflow-skinny==3.1.0" "opentelemetry-api<=1.40.0" -q

### Cell 13: Log the Readmission Model as a LoggedModel (MLflow 3)

**What's different in MLflow 3:**
- `name=` (instead of `artifact_path=`) creates a first-class LoggedModel entity
- `params=` attaches hyperparameters directly to the LoggedModel
- `mlflow.log_metrics(..., model_id=, dataset=)` links metrics to both the model AND dataset
- Compare multiple LoggedModels side-by-side in the experiment UI

> ⚠️ This cell reloads data and retrains the model since the kernel restarted.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13: Log as LoggedModel (MLflow 3 Style)
# ══════════════════════════════════════════════════════════════
# MLflow 3's LoggedModel is a first-class entity linked to its
# source run, parameters, metrics, and training datasets.
# This provides full lineage for model governance.
#
# Reference: https://learn.microsoft.com/en-us/fabric/data-science/mlflow-3-overview
# ══════════════════════════════════════════════════════════════

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from mlflow.entities import Dataset

# ── Reinitialize after kernel restart ─────────────────────────
training_spark_df = spark.table("gold_readmission_training")

feature_columns = [
    "age", "is_male", "risk_score", "is_medicare", "is_medicaid",
    "is_self_pay", "is_high_risk",
    "chronic_condition_count", "has_diabetes", "has_chf", "has_copd",
    "has_hypertension", "has_ckd", "has_depression", "has_cad",
    "has_obesity", "comorbidity_cluster_count",
    "prior_encounters_12m", "prior_admissions_12m", "prior_ed_visits_12m",
    "prior_outpatient_12m", "prior_total_charges_12m", "prior_avg_los",
    "index_los", "vitals_reading_count", "sirs_positive_count",
    "avg_heart_rate", "avg_temperature", "avg_respiratory_rate",
    "avg_spo2", "max_pain_level", "max_heart_rate", "min_spo2",
    "claim_total_amount", "avg_payment_ratio", "denied_claims_count",
    "avg_days_to_payment",
    "is_weekend_discharge", "discharge_month", "discharge_quarter",
    "is_snf_discharge", "is_home_health_discharge",
    "unique_medication_count", "is_polypharmacy"
]

pandas_df = training_spark_df.select(feature_columns + ["label"]).toPandas()
X = pandas_df[feature_columns]
y = pandas_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Retrain the model
from flaml import AutoML
automl = AutoML()
automl.fit(X_train=X_train, y_train=y_train,
           time_budget=120, metric="roc_auc", task="classification",
           estimator_list=["lgbm", "xgboost", "catboost", "rf", "extra_tree"],
           seed=42, verbose=0)

print(f"✓ Model retrained: {automl.best_estimator}")

# ── Log as a LoggedModel (MLflow 3) ──────────────────────────
mlflow.set_experiment("mlflow3-readmission-model")

with mlflow.start_run(run_name="ReadmissionModel_MLflow3") as run:
    # Wrap training data as an MLflow Dataset for lineage
    train_dataset: Dataset = mlflow.data.from_pandas(
        pd.concat([X_train, y_train], axis=1), name="readmission_train"
    )
    
    # Log the model with name= (MLflow 3 style) and params=
    model_info = mlflow.sklearn.log_model(
        sk_model=automl.model,
        name="readmission_predictor",
        params={
            "algorithm": automl.best_estimator,
            "n_features": len(feature_columns),
            "time_budget": 120,
            **{k: str(v) for k, v in automl.best_config.items()}
        },
        input_example=X_test.head(3),
    )
    
    # Retrieve the LoggedModel entity
    logged_model = mlflow.get_logged_model(model_info.model_id)
    print(f"✅ LoggedModel created:")
    print(f"   Model ID: {logged_model.model_id}")
    print(f"   Parameters: {logged_model.params}")
    
    # Compute metrics and link to LoggedModel + Dataset
    y_prob = automl.predict_proba(X_test)[:, 1]
    auc_roc = roc_auc_score(y_test, y_prob)
    
    mlflow.log_metrics(
        metrics={
            "auc_roc": auc_roc,
            "n_test_samples": len(X_test),
            "readmission_rate": float(y.mean()),
        },
        model_id=logged_model.model_id,
        dataset=train_dataset,
    )
    
    print(f"   AUC-ROC: {auc_roc:.4f}")
    print(f"   Linked to dataset: {train_dataset.name}")

print(f"\n💡 Open the experiment 'mlflow3-readmission-model' in Fabric to see:")
print(f"   • The LoggedModel in the 'Logged Models' section")
print(f"   • Parameters, metrics, and dataset linked to the model")
print(f"   • Click 'Register model' to promote it to an ML Model item")

### Cell 14: Capture Generative AI Traces

MLflow 3 traces capture the full LLM request/response lifecycle — essential for auditing AI-assisted clinical decisions:
- Complete prompt/response pairs
- Token usage (cost monitoring)
- Latency per call
- Span hierarchy for nested functions
- Error traces with stack traces

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 14: Capture Generative AI Traces
# ══════════════════════════════════════════════════════════════
# MLflow 3 traces capture every LLM call — what prompts were
# sent, what responses came back, token usage, and latency.
# This is critical for auditing AI-assisted clinical decisions.
#
# We demonstrate three trace patterns:
#   1. Feature engineering suggestion (system prompt + schema)
#   2. Clinical interpretation (model results → narrative)
#   3. Custom @mlflow.trace decorated function
#
# Reference: https://learn.microsoft.com/en-us/fabric/data-science/mlflow-3-overview
# ══════════════════════════════════════════════════════════════

import mlflow
from openai import AzureOpenAI
from synapse.ml.fabric.credentials import get_openai_httpx_sync_client

# ── Enable OpenAI autologging for traces ──────────────────────
mlflow.openai.autolog()

# Create a new AI experiment for Gen AI traces
mlflow.set_experiment("mlflow3-readmission-genai-traces")

client = AzureOpenAI(
    api_version="2025-04-01-preview",
    http_client=get_openai_httpx_sync_client(),
)

# ── Trace 1: Feature Engineering Suggestion ───────────────────
print("Capturing traces...\n")

with mlflow.start_run(run_name="feature_engineering_trace"):
    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": "You are a clinical data scientist. Suggest 5 key features for readmission prediction."},
            {"role": "user", "content": "We have patient demographics, encounter history, conditions, and vitals data. What are the top 5 features to predict 30-day readmission?"},
        ],
        temperature=0.3,
        max_completion_tokens=500,
    )
    
    trace_id = mlflow.get_last_active_trace_id()
    print(f"✅ Feature engineering trace captured")
    print(f"   Trace ID: {trace_id}")
    print(f"   Response: {response.choices[0].message.content[:200]}...")

# ── Trace 2: Clinical Interpretation ──────────────────────────
with mlflow.start_run(run_name="clinical_interpretation_trace"):
    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": "You are a clinical informatics expert. Interpret ML model results for a hospital CMO."},
            {"role": "user", "content": f"Our readmission model achieved AUC-ROC of {auc_roc:.4f} using {automl.best_estimator}. The top features are prior_admissions_12m, chronic_condition_count, and index_los. Provide a 2-sentence clinical interpretation."},
        ],
        temperature=0.4,
        max_completion_tokens=300,
    )
    
    trace_id = mlflow.get_last_active_trace_id()
    print(f"\n✅ Clinical interpretation trace captured")
    print(f"   Trace ID: {trace_id}")
    print(f"   Response: {response.choices[0].message.content[:200]}...")

# ── Trace 3: Custom traced function ──────────────────────────
@mlflow.trace
def assess_patient_risk(patient_summary: str) -> str:
    """Use AI to generate a narrative risk assessment for a patient."""
    mlflow.update_current_trace(tags={
        "use_case": "patient_risk_narrative",
        "model": "gpt-5.1",
    })
    
    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": "Generate a brief clinical risk narrative for care coordinators."},
            {"role": "user", "content": patient_summary},
        ],
        temperature=0.3,
        max_completion_tokens=200,
    )
    return response.choices[0].message.content

with mlflow.start_run(run_name="patient_risk_narrative_trace"):
    narrative = assess_patient_risk(
        "72-year-old male with CHF, COPD, and 3 prior admissions in 12 months. "
        "Readmission risk score: 0.78 (High). Being discharged to home."
    )
    print(f"\n✅ Patient risk narrative trace captured")
    print(f"   Narrative: {narrative[:200]}...")

print(f"\n{'='*60}")
print("📋 VIEW YOUR TRACES:")
print(f"{'='*60}")
print("1. Open the experiment 'mlflow3-readmission-genai-traces' in Fabric")
print("2. Click the 'Traces' tab")
print("3. Select any trace to see:")
print("   • Full prompt/response pairs")
print("   • Token usage (input + output tokens)")
print("   • Latency per LLM call")
print("   • Span hierarchy for nested function calls")
print("   • Model metadata (name, version, parameters)")

---
## Optional Part G: Monitor ML Experiments and Model Endpoints

In a clinical setting, model governance is critical:
- **Audit trail** — Who trained what model, when, with what parameters
- **Endpoint health** — Request volume, error rates, latency
- **Capacity planning** — CU consumption from active endpoints

### Cell 15: Programmatic Monitoring

Query experiments and registered models via the MLflow API. For visual monitoring, use the **Fabric Monitoring hub** (left nav → Monitor).

| What to Monitor | Where | Frequency |
|----------------|-------|----------|
| Experiment runs | Monitoring hub → Experiment filter | After each training |
| Endpoint traffic | Model → Version → Endpoint metrics | Weekly |
| Capacity usage | Fabric Capacity Metrics app | Monthly |

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 15: Monitor Experiments and Models
# ══════════════════════════════════════════════════════════════
# Programmatically query experiment runs and registered models.
# For visual monitoring, use the Fabric Monitoring hub:
#   Left nav → Monitor → Experiment filter
#
# Reference: https://learn.microsoft.com/en-us/fabric/data-science/monitor-machine-learning-experiments-models
# ══════════════════════════════════════════════════════════════

import mlflow
from pprint import pprint
from mlflow import MlflowClient

# ── View experiment runs programmatically ─────────────────────
client = MlflowClient()

print("📊 EXPERIMENT RUNS SUMMARY")
print("=" * 60)

# List all experiments in the workspace
experiments = client.search_experiments()
for exp in experiments:
    if "readmission" in exp.name.lower() or "automl" in exp.name.lower():
        print(f"\n📁 Experiment: {exp.name}")
        print(f"   ID: {exp.experiment_id}")
        
        # Get runs for this experiment
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=["start_time DESC"],
            max_results=5,
        )
        
        print(f"   Recent runs ({len(runs)}):")
        for run in runs:
            status = run.info.status
            duration = ""
            if run.info.end_time and run.info.start_time:
                dur_s = (run.info.end_time - run.info.start_time) / 1000
                duration = f" ({dur_s:.1f}s)"
            
            metrics_str = ""
            if "auc_roc" in run.data.metrics:
                metrics_str = f" | AUC-ROC: {run.data.metrics['auc_roc']:.4f}"
            
            print(f"     • {run.info.run_name}: {status}{duration}{metrics_str}")

# ── View registered models ────────────────────────────────────
print(f"\n\n📦 REGISTERED MODELS")
print("=" * 60)

for rm in client.search_registered_models():
    print(f"\n  Model: {rm.name}")
    for mv in rm.latest_versions:
        print(f"    Version {mv.version}: status={mv.status}, stage={mv.current_stage}")
        print(f"    Run ID: {mv.run_id}")

print(f"\n\n{'='*60}")
print("📋 MONITORING BEST PRACTICES FOR CLINICAL ML MODELS")
print(f"{'='*60}")
print("""  
  | Practice                    | Action                                    | Frequency |
  |----------------------------|-------------------------------------------|----------|
  | Model performance audit    | Compare AUC-ROC against training baseline | Monthly  |
  | Endpoint health check      | Review error rates and latency            | Weekly   |
  | Experiment cleanup         | Archive old experiments                   | Quarterly|
  | Capacity review            | Check CU consumption for endpoints        | Monthly  |
  | Retraining trigger         | Retrain if AUC-ROC drops >5%              | As needed|
""")

print("\n✅ Use the Fabric Monitoring hub for a visual overview:")
print("   Left navigation → Monitor → filter by 'Experiment'")
print("\n💡 Endpoint traffic metrics appear within 15 min of receiving requests.")
print("   View under: Model → Version → Endpoint metrics section")

---
## ✅ Module 7 Complete — Summary

### What You Built

| Component | Description |
|-----------|-------------|
| **Gen AI Feature Discovery** | Used Azure OpenAI in Fabric to suggest clinically-grounded features |
| **Feature Engineering Pipeline** | Built 42 features from 7 Silver/Gold tables using PySpark |
| **AutoML Model Selection** | FLAML automatically tried 5 algorithms with hyperparameter tuning |
| **MLflow Experiment Tracking** | Every trial logged for reproducibility and comparison |
| **Risk Scoring System** | Every patient scored 0.0–1.0 and classified Low/Medium/High |
| **Clinical Interpretation** | AI-generated insights for non-technical stakeholders |
| **Model Deployment** | Registered model with real-time endpoint capability |
| **MLflow 3 LoggedModel** | First-class model entity with dataset lineage |
| **Gen AI Traces** | Full audit trail of LLM calls (prompts, responses, tokens) |
| **Monitoring** | Programmatic + UI-based experiment and endpoint monitoring |

### Key Tables Created
- `gold_readmission_training` — 42-feature training dataset
- `gold_readmission_risk_scores` — Patient risk scores and tiers

### How This Connects to Other Modules
- **Module 3** (Power BI): Add `gold_readmission_risk_scores` to your semantic model
- **Module 5** (Data Agent): Query "Which high-risk patients are being discharged this week?"
- **Module 8** (VS Code Agent): Build on this model programmatically